# Fine-tune, experiments, benchmarks, release (weeks 5–9)

In [ ]:
# Settings: Accelerator = GPU T4 x2 (we use one), Internet = ON, Persistence = Files
REPO = "https://github.com/preranam05/loopthink"   # <- after you push this repo
!git clone -q $REPO /kaggle/working/loopthink || (cd /kaggle/working/loopthink && git pull -q)
%cd /kaggle/working/loopthink
!pip -q install tokenizers datasets huggingface_hub
import torch; print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
DATA='/kaggle/input/fineweb-tok-1b'
PRE={'looped':'/kaggle/input/loopthink-pretrain-looped/final.pt','dense6':'/kaggle/input/loopthink-pretrain-dense6/final.pt','dense16':'/kaggle/input/loopthink-pretrain-dense16/final.pt'}
W='/kaggle/working/runs'

## Fine-tune all models on both setups, then run every experiment

In [ ]:
for name, ck in PRE.items():
    for setup in ('official','heldout'):
        run=f'{W}/{name}_{setup}'
        !python -m loopthink.finetune --init $ck --tokenizer $DATA/tokenizer.json --setup $setup --out $run --epochs 10
        !python -m loopthink.evaluate --run $run
!python scripts/summarize.py $W/* | tee $W/results_table.md

## Latency / throughput / cost (GPU)

In [ ]:
!python -m loopthink.bench --runs $W/looped_official $W/dense6_official $W/dense16_official --taus 0.3 0.5 0.7 --gpu_hour_usd 0.35 --out $W/bench_gpu.json

## Release — set HF_TOKEN as a Kaggle secret first

In [ ]:
from kaggle_secrets import UserSecretsClient; import os
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
!python scripts/push_to_hub.py --run $W/looped_official --repo YOUR_HF_USER/loopthink-27m --space YOUR_HF_USER/loopthink-demo